# DeepSets vs Graph Neural Network Comparison

## Research-Backed Methodology

This notebook compares **all 16 training dataset splits** for both DeepSets and GraphSAGE models.

**Model Names (16 total):**
- a1_d3_00, a2_d3_10, a3_d3_20, a4_d3_40, a5_d3_50
- b1_d5heavy, b2_d5more, b3_balanced, b4_d7more, b5_d7heavy
- c1_only_d3, c2_only_d5, c3_only_d7, c4_no_d7
- equal_333333

**Distances Analyzed:** d=3, 5, 7, 9, 11, 13

In [ ]:
import sys
import json
import time
import gc
import os
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from scipy import stats
from scipy.stats import wilcoxon
import stim

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..').resolve()

sys.path.insert(0, str(BASE_PATH))

from benchmark_models import DeepSets, SurfaceCodeSampler
from models import GraphSAGE, SparseGraph

RESULTS_DIR = Path(".").resolve()
PLOTS_DIR = RESULTS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_PLOTS_DIR = PLOTS_DIR / "final"
FINAL_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"BASE_PATH: {BASE_PATH}")

In [ ]:
# ============================================================
# DATA MODE CONFIGURATION
# ============================================================
USE_SAVED_DATA = False

# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================
DISTANCES = [3, 5, 7, 9, 11, 13]

# All 16 model names (no checkpoint files)
MODEL_NAMES = [
    "a1_d3_00", "a2_d3_10", "a3_d3_20", "a4_d3_40", "a5_d3_50",
    "b1_d5heavy", "b2_d5more", "b3_balanced", "b4_d7more", "b5_d7heavy",
    "c1_only_d3", "c2_only_d5", "c3_only_d7", "c4_no_d7",
    "equal_333333"
]

TEST_SAMPLES_PER_DISTANCE = 10000
BENCHMARK_SAMPLES = 10000
BATCH_SIZES = [1, 8, 16, 32, 64, 128, 256]
NUM_WARMUP_RUNS = 10
NUM_BENCHMARK_RUNS = 5
NUM_ACCURACY_RUNS = 4
ALPHA = 0.05
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Configuration:")
print(f"  USE_SAVED_DATA: {USE_SAVED_DATA}")
print(f"  Distances: {DISTANCES}")
print(f"  Models to compare: {len(MODEL_NAMES)}")
print(f"  Test samples per distance: {TEST_SAMPLES_PER_DISTANCE:,}")

In [ ]:
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model_size_mb(model_path: Path) -> float:
    if model_path.exists():
        return model_path.stat().st_size / (1024 * 1024)
    return 0.0

def load_deepsets_model(model_name: str, base_path: Path):
    """Load DeepSets model from checkpoint."""
    model_path = base_path / "deepsets" / "extrapolation" / "models" / "revised_training" / f"{model_name}.pt"
    if not model_path.exists():
        print(f"  ✗ DeepSets {model_name} not found")
        return None, {}
    
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    config = checkpoint.get('config', {})
    
    model = DeepSets(
        nickname=model_name,
        phi_hidden=config.get('phi_hidden', [64, 128, 256]),
        rho_hidden=config.get('rho_hidden', [256, 128, 64]),
        pool=config.get('pool', 'mean'),
        dropout=config.get('dropout', 0.1),
        device=device,
        base_path=base_path
    )
    model.model.load_state_dict(checkpoint['state_dict'])
    model.model.eval()
    model._config = config
    
    return model, {
        'name': model_name,
        'num_parameters': count_parameters(model.model),
        'model_size_mb': get_model_size_mb(model_path),
        'config': config
    }

def load_graphsage_model(model_name: str, base_path: Path):
    """Load GraphSAGE model from checkpoint."""
    model_path = base_path / "gSAGE" / "extrapolation" / "models" / "revised_training" / f"{model_name}.pt"
    if not model_path.exists():
        print(f"  ✗ GraphSAGE {model_name} not found")
        return None, {}
    
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    config = checkpoint.get('config', {
        'in_channels': 5, 'hidden_dim': 128, 'num_layers': 5, 'dropout': 0.0, 'aggr': 'max'
    })
    
    model = GraphSAGE(
        nickname=model_name,
        in_channels=config.get('in_channels', 5),
        hidden_dim=config.get('hidden_dim', 128),
        num_layers=config.get('num_layers', 5),
        dropout=config.get('dropout', 0.0),
        aggr=config.get('aggr', 'max'),
        device=device,
        base_path=base_path
    )
    if 'state_dict' in checkpoint:
        model.model.load_state_dict(checkpoint['state_dict'])
    model.model.eval()
    
    return model, {
        'name': model_name,
        'num_parameters': count_parameters(model.model),
        'model_size_mb': get_model_size_mb(model_path),
        'config': config
    }

print("Helper functions defined.")

In [ ]:
# Load ALL 16 model pairs
print("Loading all 16 model pairs...\n")

deepsets_models = {}
graphsage_models = {}
deepsets_infos = {}
graphsage_infos = {}

for name in MODEL_NAMES:
    print(f"Loading {name}...")
    
    ds_model, ds_info = load_deepsets_model(name, BASE_PATH)
    if ds_model:
        deepsets_models[name] = ds_model
        deepsets_infos[name] = ds_info
        print(f"  ✓ DeepSets: {ds_info['num_parameters']:,} params")
    
    gs_model, gs_info = load_graphsage_model(name, BASE_PATH)
    if gs_model:
        graphsage_models[name] = gs_model
        graphsage_infos[name] = gs_info
        print(f"  ✓ GraphSAGE: {gs_info['num_parameters']:,} params")

print(f"\nLoaded {len(deepsets_models)} DeepSets models")
print(f"Loaded {len(graphsage_models)} GraphSAGE models")

In [ ]:
# Generate shared test datasets
shared_test_data = {}
graph_builder = SparseGraph(k_neighbors=6, device=torch.device('cpu'))
sampler = SurfaceCodeSampler(p=0.005, device=torch.device('cpu'))

print("Generating shared test datasets...")
for d in DISTANCES:
    torch.manual_seed(SEED + d)
    np.random.seed(SEED + d)
    
    detections, labels = sampler.sample(
        d=d,
        num_samples=TEST_SAMPLES_PER_DISTANCE,
        p_values=[0.001, 0.003, 0.005, 0.007],
        p_weights=[0.25, 0.25, 0.25, 0.25]
    )
    
    graphs = graph_builder.batch_to_pyg(detections, labels)
    
    shared_test_data[d] = {
        'detections': detections.cpu(),
        'labels': labels.cpu(),
        'graphs': graphs,
        'num_samples': len(labels)
    }
    print(f"  ✓ d={d}: {len(labels):,} samples")

print(f"\nGenerated test data for {len(shared_test_data)} distances")

In [ ]:
def evaluate_model_accuracy(model, data, labels, is_graph=False, is_deepsets=False, distance=None, batch_size=256):
    """Evaluate accuracy for a single model."""
    model.model.eval()
    model.model.to(device)
    
    all_accuracies = []
    
    for run in range(NUM_ACCURACY_RUNS):
        with torch.no_grad():
            predictions = []
            
            if is_graph:
                loader = DataLoader(data, batch_size=batch_size, shuffle=False)
                for batch in loader:
                    batch = batch.to(device)
                    pred = model.model(batch)
                    predictions.append(pred.cpu())
            elif is_deepsets:
                for i in range(0, len(data), batch_size):
                    batch = data[i:i+batch_size].to(device)
                    pred = model.predict(batch, distance)
                    predictions.append(pred.cpu())
            
            predictions = torch.cat(predictions, dim=0).squeeze()
            binary_preds = (predictions >= 0.5).float()
            acc = (binary_preds == labels.float()).float().mean().item()
            all_accuracies.append(acc)
    
    return {
        'accuracy_mean': np.mean(all_accuracies),
        'accuracy_std': np.std(all_accuracies),
        'all_accuracies': all_accuracies
    }

# Evaluate all models on all distances
print("Evaluating all models...\n")
accuracy_results = {}

for model_name in tqdm(MODEL_NAMES, desc="Models"):
    accuracy_results[model_name] = {}
    
    for d in DISTANCES:
        accuracy_results[model_name][d] = {}
        labels = shared_test_data[d]['labels']
        
        # DeepSets
        if model_name in deepsets_models:
            detections = shared_test_data[d]['detections']
            results = evaluate_model_accuracy(
                deepsets_models[model_name], detections, labels,
                is_deepsets=True, distance=d
            )
            accuracy_results[model_name][d]['deepsets'] = results
        
        # GraphSAGE
        if model_name in graphsage_models:
            graphs = shared_test_data[d]['graphs']
            results = evaluate_model_accuracy(
                graphsage_models[model_name], graphs, labels,
                is_graph=True
            )
            accuracy_results[model_name][d]['graphsage'] = results

print("\n✓ Accuracy evaluation complete")

In [ ]:
# Statistical Tests - Wilcoxon for each model
print("Performing statistical tests...\n")
statistical_tests = {}

for model_name in MODEL_NAMES:
    ds_all = []
    gs_all = []
    
    for d in DISTANCES:
        if d in accuracy_results.get(model_name, {}):
            if 'deepsets' in accuracy_results[model_name][d]:
                ds_all.extend(accuracy_results[model_name][d]['deepsets']['all_accuracies'])
            if 'graphsage' in accuracy_results[model_name][d]:
                gs_all.extend(accuracy_results[model_name][d]['graphsage']['all_accuracies'])
    
    if len(ds_all) >= 2 and len(gs_all) >= 2 and len(ds_all) == len(gs_all):
        stat, p_val = wilcoxon(ds_all, gs_all, alternative='two-sided')
        statistical_tests[model_name] = {
            'statistic': float(stat),
            'p_value': float(p_val),
            'significant': p_val < ALPHA,
            'ds_mean': np.mean(ds_all),
            'gs_mean': np.mean(gs_all),
            'diff': np.mean(ds_all) - np.mean(gs_all)
        }
        print(f"{model_name}: DS={np.mean(ds_all):.4f}, GS={np.mean(gs_all):.4f}, p={p_val:.4f}")

print("\n✓ Statistical tests complete")

In [ ]:
# Create summary DataFrame
summary_data = []

for model_name in MODEL_NAMES:
    for d in DISTANCES:
        if model_name in accuracy_results and d in accuracy_results[model_name]:
            ds_acc = accuracy_results[model_name][d].get('deepsets', {}).get('accuracy_mean', np.nan)
            gs_acc = accuracy_results[model_name][d].get('graphsage', {}).get('accuracy_mean', np.nan)
            
            summary_data.append({
                'model': model_name,
                'distance': d,
                'deepsets_acc': ds_acc,
                'graphsage_acc': gs_acc,
                'diff': ds_acc - gs_acc if not np.isnan(ds_acc) and not np.isnan(gs_acc) else np.nan
            })

summary_df = pd.DataFrame(summary_data)
print("Summary Table:")
print(summary_df.pivot(index='model', columns='distance', values='diff').to_string())

In [ ]:
# Plot: Accuracy Comparison Heatmap (DeepSets - GraphSAGE difference)
pivot_diff = summary_df.pivot(index='model', columns='distance', values='diff')

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(pivot_diff, annot=True, fmt='.3f', cmap='RdYlGn', center=0, ax=ax)
ax.set_title('Accuracy Difference (DeepSets - GraphSAGE)\nGreen = DeepSets better, Red = GraphSAGE better', fontsize=14)
ax.set_xlabel('Code Distance', fontsize=12)
ax.set_ylabel('Model Name', fontsize=12)
plt.tight_layout()
plt.savefig(FINAL_PLOTS_DIR / 'accuracy_difference_heatmap.png', dpi=150)
plt.show()
print(f"Saved: {FINAL_PLOTS_DIR / 'accuracy_difference_heatmap.png'}")

In [ ]:
# Plot: Wilcoxon Results
wilcoxon_data = []
for model_name, res in statistical_tests.items():
    wilcoxon_data.append({
        'model': model_name,
        'p_value': res['p_value'],
        'significant': res['significant'],
        'ds_mean': res['ds_mean'],
        'gs_mean': res['gs_mean'],
        'diff': res['diff']
    })

wilcoxon_df = pd.DataFrame(wilcoxon_data)
wilcoxon_df = wilcoxon_df.sort_values('diff', ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['green' if d > 0 else 'red' for d in wilcoxon_df['diff']]
bars = ax.barh(wilcoxon_df['model'], wilcoxon_df['diff'], color=colors, alpha=0.7)
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Mean Accuracy Difference (DeepSets - GraphSAGE)', fontsize=12)
ax.set_ylabel('Model', fontsize=12)
ax.set_title('DeepSets vs GraphSAGE: Per-Model Performance Difference', fontsize=14)

# Add significance markers
for i, (idx, row) in enumerate(wilcoxon_df.iterrows()):
    if row['significant']:
        ax.annotate('*', xy=(row['diff'], i), fontsize=16, ha='center', va='bottom')

plt.tight_layout()
plt.savefig(FINAL_PLOTS_DIR / 'model_comparison_barplot.png', dpi=150)
plt.show()
print(f"Saved: {FINAL_PLOTS_DIR / 'model_comparison_barplot.png'}")

In [ ]:
# Save results
summary_df.to_csv(FINAL_PLOTS_DIR / 'final_accuracies.csv', index=False)
wilcoxon_df.to_csv(FINAL_PLOTS_DIR / 'wilcoxon_results.csv', index=False)

print(f"Saved: {FINAL_PLOTS_DIR / 'final_accuracies.csv'}")
print(f"Saved: {FINAL_PLOTS_DIR / 'wilcoxon_results.csv'}")

# Summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
ds_wins = (wilcoxon_df['diff'] > 0).sum()
gs_wins = (wilcoxon_df['diff'] < 0).sum()
print(f"DeepSets better in {ds_wins}/{len(wilcoxon_df)} models")
print(f"GraphSAGE better in {gs_wins}/{len(wilcoxon_df)} models")
print(f"Average difference: {wilcoxon_df['diff'].mean():.4f}")